# Output Parsers — Cómo forzar el formato de salida de un LLM

> **Objetivo:** Aprender a controlar exactamente cómo responde el modelo: texto plano, JSON o objetos Python validados.

## El problema que resuelven los Output Parsers

Por defecto, un LLM responde como quiere:
```
Prompt: "Dame información sobre Madrid en JSON"
Respuesta: "¡Claro! Aquí tienes la información:\n```json\n{\"ciudad\": \"Madrid\", ...}\n```"
```
Para usar eso en código, tendrías que extraer el JSON manualmente. Es frágil y poco profesional.

**Con un Output Parser:**
```python
resultado = cadena.invoke(...)  # → {"ciudad": "Madrid", ...}  ← dict Python directamente
```

| Sin Parser | Con Parser |
|-----------|------------|
| Respuesta en texto libre | Respuesta en formato garantizado |
| Necesitas parsear manualmente | Conversión automática |
| Frágil (el modelo puede cambiar el formato) | Robusto (si el formato falla, da error claro) |

---

## 0. Preparación

In [ ]:
# Instalamos todas las dependencias necesarias
!pip install langchain langchain-google-genai python-dotenv pydantic -q
!pip install langchain
!pip install langchain-community

In [14]:
import os
import json
from typing import List, Optional

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain.output_parsers import OutputFixingParser
from langchain_core.output_parsers import PydanticOutputParser#, OutputFixingParser
from pydantic import BaseModel, Field, field_validator
from dotenv import load_dotenv

# Carga credenciales
load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")
API_KEY = os.getenv("Gemini_API_Key_001")

# Modelo base para todos los ejemplos
MODEL = "gemini-2.5-flash-lite"
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.5, google_api_key=API_KEY)

print("✅ Entorno listo")

ModuleNotFoundError: No module named 'langchain.output_parsers'

## 1. StrOutputParser — El Más Simple

Extrae el texto de la respuesta del modelo y lo devuelve como un `str` Python.

### ¿Por qué usarlo si parece trivial?
Sin él, `llm.invoke()` devuelve un objeto `AIMessage`. Con él, devuelves directamente el string.

```python
# Sin StrOutputParser
resp = llm.invoke("Hola")  # → AIMessage(content='Hola', ...)
texto = resp.content        # → 'Hola'

# Con StrOutputParser
cadena = llm | StrOutputParser()
texto = cadena.invoke("Hola")  # → 'Hola' directamente
```

En chains largas, es fundamental porque los parsers posteriores esperan strings.

In [ ]:
# Comparación sin parser vs con StrOutputParser

# Sin parser: devuelve AIMessage (objeto)
resp_raw = llm.invoke("Di hola en 3 idiomas.")
print(f"Sin parser → tipo: {type(resp_raw).__name__}")
print(f"  resp_raw.content: {resp_raw.content}")

# Con StrOutputParser: devuelve string directamente
cadena = ChatPromptTemplate.from_messages([
    ("human", "{input}")
]) | llm | StrOutputParser()

resp_str = cadena.invoke({"input": "Di hola en 3 idiomas."})
print(f"\nCon StrOutputParser → tipo: {type(resp_str).__name__}")
print(f"  valor: {resp_str}")

## 2. JsonOutputParser — Extraer JSON

Convierte la respuesta del modelo en un `dict` (o `list`) Python.

### Lo que hace internamente:
1. Recibe el texto del modelo (puede incluir ```json ... ```)
2. Extrae el JSON puro (elimina markdown)
3. Llama a `json.loads()` para convertirlo en dict
4. Devuelve el dict

### Cuándo usarlo:
- Cuando el modelo debe devolver datos estructurados
- Cuando no necesitas validar los tipos o campos específicos
- Cuando la estructura puede variar entre respuestas

In [ ]:
# JsonOutputParser sin schema — acepta cualquier JSON válido
parser_json = JsonOutputParser()

template_json = ChatPromptTemplate.from_messages([
    ("system", "Siempre responde con un JSON válido sin markdown ni backticks."),
    ("human", "{pregunta}")
])

# Cadena: template genera el prompt → llm responde → parser convierte a dict
cadena_json = template_json | llm | parser_json

resultado = cadena_json.invoke({
    "pregunta": "Dame información sobre Madrid: población, país, fundación y famoso por qué."
})

print(f"Tipo resultado: {type(resultado).__name__}")  # dict
print("\nContenido:")
print(json.dumps(resultado, ensure_ascii=False, indent=2))

In [ ]:
# JsonOutputParser es robusto: maneja automáticamente respuestas con ```json ... ```
template_con_markdown = ChatPromptTemplate.from_messages([
    ("human", "Dame información básica sobre Python (el lenguaje) en formato JSON")
])

try:
    # Aunque el modelo responda con ```json ... ```, el parser lo limpia automáticamente
    resultado = (template_con_markdown | llm | parser_json).invoke({})
    print("✅ Parseado correctamente:")
    print(json.dumps(resultado, ensure_ascii=False, indent=2))
except Exception as e:
    print(f"❌ Error de parseo: {e}")

## 3. PydanticOutputParser — Schema Estricto con Validación

Es el parser más potente. Combina JSON con **validación de tipos y valores**.

### Cómo funciona:
```
1. Defines un schema Pydantic (qué campos, de qué tipo, con qué restricciones)
2. El parser genera instrucciones de formato para incluir en el prompt
3. El modelo responde en JSON siguiendo esas instrucciones
4. El parser convierte el JSON en un objeto Python tipado y lo valida
```

### Parámetros de Field():
- `description`: explica al modelo qué debe poner en ese campo
- `ge`, `le`: validaciones numéricas (greater_equal, less_equal)
- `default`: valor por defecto si el modelo no lo incluye

In [ ]:
# Schema Pydantic — define la estructura EXACTA de la respuesta
class AnalisisSentimiento(BaseModel):
    clasificacion: str = Field(
        description="Clasificación del sentimiento: POSITIVO, NEGATIVO o MIXTO"
    )
    puntuacion: int = Field(
        description="Puntuación del 1 al 5, donde 1=muy negativo y 5=muy positivo",
        ge=1, le=5   # Pydantic valida: si el modelo pone 0 o 6, da error
    )
    aspectos_positivos: List[str] = Field(
        description="Lista de aspectos positivos mencionados en la reseña"
    )
    aspectos_negativos: List[str] = Field(
        description="Lista de aspectos negativos mencionados en la reseña"
    )
    recomendaria: bool = Field(
        description="Si el cliente recomendaría el producto basándose en la reseña"
    )
    resumen: str = Field(
        description="Resumen breve de la reseña en máximo 20 palabras"
    )

    # Validador personalizado: lógica extra que Pydantic ejecuta al parsear
    @field_validator('clasificacion')
    @classmethod
    def validar_clasificacion(cls, v):
        permitidos = {'POSITIVO', 'NEGATIVO', 'MIXTO'}
        if v.upper() not in permitidos:
            raise ValueError(f"Clasificación debe ser una de: {permitidos}")
        return v.upper()  # Normaliza a mayúsculas siempre

In [ ]:
parser_pydantic = PydanticOutputParser(pydantic_object=AnalisisSentimiento)

# get_format_instructions() genera el texto que le dice al modelo exactamente qué JSON debe producir
# Es fundamental incluirlo en el system prompt
print("Instrucciones de formato generadas por el parser:")
print("-" * 50)
print(parser_pydantic.get_format_instructions())

In [ ]:
# El template incluye {format_instructions} que se rellena con el schema
template_pydantic = ChatPromptTemplate.from_messages([
    ("system", """Eres un analizador de reseñas de productos.

{format_instructions}"""),
    ("human", "Analiza esta reseña:\n\n{resena}")
])

cadena_pydantic = template_pydantic | llm | parser_pydantic

RESENA_COMPLEJA = """
Compré este robot de cocina hace 3 meses y tengo sentimientos encontrados.
La potencia es brutal y pica verduras perfectamente, pero el ruido es ensordecedor.
El material parece de calidad y el diseño es elegante. Sin embargo, la app es un desastre
absoluto, se cuelga constantemente. El servicio técnico fue rapidísimo cuando reporté el problema.
Por el precio (450€) esperaba más, especialmente en software. El hardware, un 9; el software, un 3.
"""

resultado = cadena_pydantic.invoke({
    "resena": RESENA_COMPLEJA,
    "format_instructions": parser_pydantic.get_format_instructions()
})

# Acceso con notación de punto (objeto Python, no dict)
print(f"Tipo resultado: {type(resultado).__name__}")  # AnalisisSentimiento
print(f"\nClasificación: {resultado.clasificacion}")
print(f"Puntuación: {resultado.puntuacion}/5")
print(f"¿Recomendaría?: {'Sí' if resultado.recomendaria else 'No'}")
print(f"\nAspectos positivos:")
for asp in resultado.aspectos_positivos:
    print(f"  ✅ {asp}")
print(f"\nAspectos negativos:")
for asp in resultado.aspectos_negativos:
    print(f"  ❌ {asp}")
print(f"\nResumen: {resultado.resumen}")

## 4. Schemas Anidados con Pydantic

Pydantic permite estructuras complejas con **modelos dentro de modelos** (composición).

### Cuándo usar schemas anidados:
- Cuando un campo es a su vez un objeto con atributos (ej: una empresa tiene nombre, sector, país)
- Para representar relaciones jerárquicas en la respuesta
- Extracción de información estructurada de textos

In [ ]:
# Modelos anidados: Persona y Empresa son modelos que se usan dentro de ExtraccionNoticia
class Persona(BaseModel):
    nombre: str = Field(description="Nombre completo de la persona")
    edad: Optional[int] = Field(default=None, description="Edad en años, si se menciona")
    ocupacion: Optional[str] = Field(default=None, description="Ocupación o cargo")

class Empresa(BaseModel):
    nombre: str = Field(description="Nombre de la empresa")
    sector: str = Field(description="Sector o industria de la empresa")
    pais: Optional[str] = Field(default=None, description="País donde opera")

class ExtraccionNoticia(BaseModel):
    titulo_sugerido: str = Field(description="Título breve sugerido para la noticia")
    personas: List[Persona] = Field(description="Personas mencionadas en el texto")
    empresas: List[Empresa] = Field(description="Empresas mencionadas en el texto")
    fecha_evento: Optional[str] = Field(default=None, description="Fecha del evento si se menciona")
    cifras_clave: List[str] = Field(description="Cifras o datos numéricos importantes")
    palabras_clave: List[str] = Field(description="5 palabras clave del texto")

parser_noticia = PydanticOutputParser(pydantic_object=ExtraccionNoticia)

template_extraccion = ChatPromptTemplate.from_messages([
    ("system", "Extrae información estructurada del texto. {format_instructions}"),
    ("human", "{texto}")
])

cadena_extraccion = template_extraccion | llm | parser_noticia

NOTICIA = """
La startup madrileña DataCore, fundada en 2022 por María López (CEO, 34 años) y
Javier Sánchez (CTO, 38 años), ha cerrado una ronda Serie A de 8 millones de euros
liderada por el fondo alemán TechVentures. La empresa, especializada en análisis
de datos para el sector retail, cuenta ya con 45 empleados y espera alcanzar los
3 millones de euros de ARR a finales de 2024. El capital se destinará principalmente
a la expansión internacional hacia Francia e Italia.
"""

resultado = cadena_extraccion.invoke({
    "texto": NOTICIA,
    "format_instructions": parser_noticia.get_format_instructions()
})

print(f"📰 Título: {resultado.titulo_sugerido}")
print(f"\n👥 Personas:")
for p in resultado.personas:
    edad_str = f", {p.edad} años" if p.edad else ""
    print(f"  • {p.nombre}{edad_str} — {p.ocupacion}")
print(f"\n🏢 Empresas:")
for e in resultado.empresas:
    print(f"  • {e.nombre} ({e.sector}, {e.pais})")
print(f"\n📊 Cifras clave: {', '.join(resultado.cifras_clave)}")
print(f"🔑 Palabras clave: {', '.join(resultado.palabras_clave)}")

## 5. OutputFixingParser — Reparar Respuestas Inválidas

A veces el modelo no sigue el formato perfectamente. `OutputFixingParser` usa **otro LLM** para corregir la respuesta antes de parsearla.

### Flujo:
```
Respuesta mal formateada
    → PydanticOutputParser intenta parsear → falla
    → OutputFixingParser envía el error + la respuesta al LLM
    → El LLM devuelve la versión corregida
    → PydanticOutputParser parsea la versión corregida → éxito
```

### Cuándo usarlo:
- Como fallback en pipelines de producción
- Cuando usas modelos menos capaces que pueden fallar el formato
- Nunca como primera opción (tiene coste adicional de llamada al LLM)

In [ ]:
# Simulamos una respuesta mal formateada del modelo
respuesta_mal_formateada = """
Aquí está mi análisis:

- clasificacion: positivo (con minúscula y sin comillas)
- puntuacion: "cuatro" (string en lugar de int)
- aspectos_positivos: [buena calidad, envío rápido]
- aspectos_negativos: ninguno
- recomendaria: sí
- resumen: producto muy bueno
"""

print("Intentando parsear respuesta mal formateada...")
try:
    resultado_directo = parser_pydantic.parse(respuesta_mal_formateada)
    print("✅ Parseado OK")
except Exception as e:
    # Este error es ESPERADO: el parser no puede convertir "cuatro" a int
    print(f"❌ Error esperado: {type(e).__name__}")
    print(f"   {str(e)[:200]}")

In [ ]:
# OutputFixingParser: envuelve al parser base y añade capacidad de auto-corrección
fixing_parser = OutputFixingParser.from_llm(
    parser=parser_pydantic,  # El parser que queremos que intente usar primero
    llm=llm                  # El LLM que usará para corregir si falla
)

print("Intentando reparar con OutputFixingParser...")
try:
    resultado_corregido = fixing_parser.parse(respuesta_mal_formateada)
    print(f"✅ Reparado correctamente")
    print(f"   Clasificación: {resultado_corregido.clasificacion}")
    print(f"   Puntuación: {resultado_corregido.puntuacion}")
    print(f"   Recomendaría: {resultado_corregido.recomendaria}")
    print(f"   Resumen: {resultado_corregido.resumen}")
except Exception as e:
    print(f"❌ No pudo repararse: {e}")

## 6. Pipeline Completo de Producción

Aquí combinamos todo: PydanticParser + OutputFixingParser + reintentos.

### Patrón robusto para producción:
```
Intento 1: template | llm | PydanticParser → si falla:
Intento 2: template | llm | StrOutputParser → OutputFixingParser → si falla:
Devuelve error con información
```

In [ ]:
class ResultadoAnalisis(BaseModel):
    """Schema de salida para el análisis de reseñas."""
    clasificacion: str = Field(description="POSITIVO, NEGATIVO o MIXTO")
    puntuacion: int = Field(description="Puntuación 1-5", ge=1, le=5)
    resumen: str = Field(description="Resumen en máx 15 palabras")
    accion_recomendada: str = Field(
        description="Acción recomendada: responder_agradeciendo, responder_disculpandose, escalar_soporte, monitorizar"
    )

def analizar_resena_produccion(resena: str, reintentos: int = 2) -> dict:
    """
    Pipeline robusto de análisis de reseñas con reintentos y fallback.
    
    Args:
        resena: texto de la reseña a analizar
        reintentos: número de intentos antes de rendirse
    
    Returns:
        dict con status ('ok', 'fixed', 'error'), datos y número de intento
    """
    parser = PydanticOutputParser(pydantic_object=ResultadoAnalisis)
    fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=llm)

    template = ChatPromptTemplate.from_messages([
        ("system", """
Eres un sistema de análisis de reseñas de e-commerce.
Analiza cada reseña y proporciona un análisis estructurado.

{format_instructions}

Para accion_recomendada:
- responder_agradeciendo: reseñas positivas (4-5 estrellas)
- responder_disculpandose: reseñas negativas (1-2 estrellas)
- escalar_soporte: menciona problemas técnicos o legales
- monitorizar: reseñas mixtas o ambiguas
"""),
        ("human", "Analiza: {resena}")
    ])

    for intento in range(reintentos + 1):
        try:
            # Intento principal con PydanticParser directo
            cadena = template | llm | parser
            resultado = cadena.invoke({
                "resena": resena,
                "format_instructions": parser.get_format_instructions()
            })
            return {"status": "ok", "datos": resultado.dict(), "intento": intento + 1}

        except Exception as e:
            if intento < reintentos:
                print(f"⚠️  Intento {intento+1} fallido, intentando con fixing parser...")
                try:
                    # Fallback: obtenemos el texto raw y lo pasamos al FixingParser
                    from langchain_core.output_parsers import StrOutputParser
                    raw = (template | llm | StrOutputParser()).invoke({
                        "resena": resena,
                        "format_instructions": parser.get_format_instructions()
                    })
                    resultado = fixing_parser.parse(raw)
                    return {"status": "fixed", "datos": resultado.dict(), "intento": intento + 1}
                except:
                    pass
            else:
                return {"status": "error", "error": str(e), "intento": intento + 1}

# Test con diferentes tipos de reseñas
resenas_test = [
    "Producto excelente, llegó en 24h y superó mis expectativas. ¡Lo recomiendo!",
    "Llevo 2 semanas esperando y nadie responde a mis emails. Voy a ir a consumidores.",
    "La calidad del producto es buena pero el tiempo de entrega fue el doble de lo prometido.",
]

print("PIPELINE DE ANÁLISIS EN PRODUCCIÓN")
print("=" * 60)

for i, resena in enumerate(resenas_test, 1):
    print(f"\n[{i}] '{resena[:60]}...'")
    resultado = analizar_resena_produccion(resena)

    if resultado["status"] in ("ok", "fixed"):
        datos = resultado["datos"]
        print(f"    🏷️  {datos['clasificacion']} | ⭐ {datos['puntuacion']}/5")
        print(f"    📋 {datos['resumen']}")
        print(f"    ⚡ Acción: {datos['accion_recomendada']}")
        if resultado["status"] == "fixed":
            print(f"    ⚠️  Requirió corrección automática")
    else:
        print(f"    ❌ Error: {resultado['error']}")

## 7. Bonus: Ver el JSON Schema

Pydantic puede generar el schema JSON de tu modelo. Útil para documentación de APIs o para debuggear qué le estás pidiendo al modelo.

In [ ]:
# model_json_schema() genera el schema estándar JSON Schema de la clase Pydantic
# Es lo que subyace a las format_instructions que el parser envía al modelo
print("JSON Schema de ResultadoAnalisis:")
print(json.dumps(ResultadoAnalisis.model_json_schema(), indent=2, ensure_ascii=False))

## Resumen

| Parser | Tipo de salida | Cuándo usarlo |
|--------|---------------|---------------|
| **StrOutputParser** | `str` | Siempre como base; texto libre |
| **JsonOutputParser** | `dict` / `list` | JSON sin estructura fija |
| **PydanticOutputParser** | Instancia Pydantic | Schema exacto + validación de tipos |
| **OutputFixingParser** | Igual que el base | Fallback cuando el modelo falla el formato |

### Flujo recomendado en producción:
```
1. Define schema Pydantic
        ↓
2. Crea PydanticOutputParser  →  genera format_instructions
        ↓
3. Incluye format_instructions en el system prompt
        ↓
4. Chain: template | llm | parser
        ↓  (si falla)
5. Fallback: OutputFixingParser
```